# **🚀 NASA 항공기 엔진 잔여 수명(RUL) 예측 프로젝트**

1. 도메인 지식 기반 피처 엔지니어링 (Domain-Driven Feature Engineering)
    - 물리 지표 생성: 단순 센서 수치를 넘어 압축기 온도비(LPC/HPC TR), 연료 유량(Wf) 등 엔진의 열화 상태를 민감하게 반영하는 물리 공식을 적용하여 변수를 생성했습니다.
    - 센서 이름 최적화: sensor_11과 같은 기계적 명칭을 T24 (저압 압축기 출구 온도) 등 물리적 의미가 담긴 이름으로 변경하여 분석의 해석력을 높였습니다.

2. 시계열 데이터 최적화 전처리 (Time-Series Data Preprocessing)
    - 데이터 스무딩 (Moving Average): 센서의 미세한 진동(노이즈)을 제거하기 위해 5단계 이동 평균을 적용, 엔진 상태의 장기적인 추세를 선명하게 포착했습니다.
    - 정규화 (MinMaxScaler): 서로 다른 단위(온도, 압력 등)를 가진 변수들을 0~1 사이로 통일하여 모든 데이터가 공평하게 학습되도록 설계했습니다.

3. 데이터 누수 방지 검증 전략 (Strict Validation Strategy)
    - GroupShuffleSplit 적용: 일반적인 무작위 분할 대신 '엔진 번호'를 기준으로 데이터를 분리했습니다. 이는 특정 엔진의 과거 데이터가 시험 문제에 섞여 들어가는 '데이터 누수(Leakage)'를 원천 차단하여 모델의 실전 성능을 보장합니다.

4. 수치 데이터 무결성 확보 (Data Integrity)
    - 부동소수점 오차 정제 (clean_round): 컴퓨터 연산 과정에서 생기는 미세한 찌꺼기 숫자를 정리하여 학습의 정밀도를 높였습니다.
    - 예외 처리: 계산 과정에서 발생할 수 있는 무한대(inf)나 결측치를 안전하게 보정하여 모델의 중단 없는 학습을 가능케 했습니다.

5. 머신러닝 앙상블 기법 (Ensemble Learning)
    - 7종 어벤져스 모델 결성: 선형 회귀, 릿지, 라쏘, 의사결정나무, 랜덤 포레스트, XGBoost, SVR(서포트 벡터 회귀) 등 서로 다른 특성을 가진 7개 알고리즘을 융합했습니다.
    - 보팅(Voting) 메커니즘: 여러 전문가(모델)의 의견을 평균 내어 최종 답안을 도출함으로써, 단일 모델 사용 시 발생할 수 있는 오차를 최소화하고 예측의 안정성을 극대화했습니다.

6. 정밀 하이퍼파라미터 튜닝 (Hyperparameter Optimization)
    - 맞춤형 최적화: 기본값(Default) 모델에 의존하지 않고, 엔진 데이터의 복잡도에 맞춰 나무의 깊이(max_depth), 학습률(learning_rate), 오차 허용 범위(C, epsilon) 등을 미세 조정했습니다.
    - 과적합(Overfitting) 제어: 학습 데이터에만 매몰되지 않도록 규제 계수(alpha)와 샘플링 비율(subsample)을 설정하여, 처음 보는 엔진 데이터에 대해서도 높은 일반화 성능을 유지하도록 설계했습니다.

7. 성능 기반 가중치 보팅 (Performance-Based Weighted Voting)
    - 차등 투표권 부여: 모든 모델을 평등하게 대우하는 대신, 개별 검증 성능(RMSE)이 우수했던 XGBoost와 RandomForest 등 '에이스 모델'에 더 높은 가중치를 부여했습니다.
    - 지능적 결합: 실력이 검증된 전문가의 의견을 중시하는 전략을 통해, 단일 모델(XGBoost) 단독 성능보다 더욱 정교하고 안정적인 RMSE 12.65의 예측 정확도를 달성했습니다.

8. AutoML 기반 하이브리드 앙상블 (Advanced Hybrid Stacking)
    - 인공신경망(Deep Learning) 융합: 정통 머신러닝의 수치 해석 능력과 딥러닝(TabularNN)의 복잡 패턴 인식 능력을 결합한 하이브리드 체계를 구축했습니다.
    - 멀티 레이어 스태킹 (Multi-layer Stacking): AutoGluon을 도입하여 수천 개의 모델 조합을 스스로 탐색하게 했으며, 인공지능이 다른 인공지능의 오차를 스스로 보정하는 '스태킹' 기법을 통해 예측 성능을 프로젝트 한계치까지 끌어올렸습니다.

In [ ]:
# [1. 도구 상자 준비]
# - 쓰임새: 데이터 분석(Pandas), 수치 계산(NumPy), 시각화(Matplotlib/Seaborn), 머신러닝 모델들을 불러옵니다.
# - 적용이유: 요리를 시작하기 전 필요한 조리 도구와 양념들을 미리 선반에 꺼내두는 작업과 같습니다.
# - 변경내용: 파이썬이 데이터 분석 및 7종의 머신러닝 알고리즘을 사용할 수 있는 상태가 됩니다.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from autogluon.tabular import TabularPredictor

In [ ]:
# [2. 데이터 로드 및 이름표 설정]
# - 쓰임새: 이름표 없는 txt 데이터를 불러오고 '엔진 번호', '가동 시간', '센서 1~21' 등의 이름을 붙여줍니다.
# - 적용이유: 원본 데이터는 숫자만 나열되어 있어, 어떤 열이 온도인지 압력인지 구분하기 위해 이름표가 필수입니다.
# - 변경내용: 단순한 텍스트 파일이 우리가 읽을 수 있는 '엑셀 형태의 표(DataFrame)'로 바뀝니다.

index_names = ['unit_number', 'time_in_cycles']
setting_names = ['op_setting_1', 'op_setting_2', 'op_setting_3']
sensor_names = [f'sensor_{i}' for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

In [ ]:
# [3. 데이터 파일 불러오기]
# - 쓰임새: PC에 저장된 txt 파일을 분석 도구로 읽어와 '가상의 엑셀 표' 형태로 메모리에 올립니다.
# - 적용이유: 하드디스크에 잠자고 있는 텍스트 데이터를 분석 가능한 디지털 데이터로 변환하기 위함입니다.
# - 변경내용: 
# - sep='\s+': 공백으로 구분된 글자들을 표의 칸으로 나눕니다.
# - names=col_names: 앞서 만든 이름표를 데이터 머리에 붙여 '구조화된 표(DataFrame)'가 완성됩니다.

train_df = pd.read_csv(r"C:\Users\yuzhd\github\DataScience\scikit-learn\scikit-learn\data\CMaps\train_FD001.txt",sep='\s+',header=None,index_col=False,names=col_names)
test_df = pd.read_csv(r"C:\Users\yuzhd\github\DataScience\scikit-learn\scikit-learn\data\CMaps\test_FD001.txt",sep='\s+',header=None,index_col=False,names=col_names)
rul_df = pd.read_csv(r"C:\Users\yuzhd\github\DataScience\scikit-learn\scikit-learn\data\CMaps\RUL_FD001.txt",sep='\s+',header=None,index_col=False,names=['RUL'])

In [ ]:
# [4. 데이터 품질 확인 및 통계 분석]
# - 쓰임새: .head()로 상위 10개 데이터를 보고, .describe()로 평균/최솟값/최댓값 등 통계를 확인합니다.
# - 적용이유: 데이터가 깨지지는 않았는지, 혹은 값이 전혀 변하지 않는 고장 난 센서가 있는지 분석 전 '사전 검사'를 하는 과정입니다.
# - 변경내용: 분석가가 데이터의 전체적인 크기와 숫자 범위를 파악하게 됩니다.

train_df.head(10)

In [ ]:
# [5. 도메인 지식 기반 컬럼명 변경]
# - 쓰임새: 'sensor_2' 같은 기계적인 이름을 'T24 (저압 압축기 온도)'처럼 물리적 의미가 담긴 이름으로 바꿉니다.
# - 적용이유: 나중에 인공지능이 "어떤 센서가 가장 중요한가요?"라고 답했을 때, 분석가가 "아, 온도가 중요하군요!"라고 바로 해석하기 위함입니다.
# - 변경내용: 모델의 해석력이 높아지며, 비전공자나 현장 전문가도 이해할 수 있는 '직관적인 표'가 됩니다.

rename_dict = {
    "op_setting_1": "Alt[kft]", # Altitude
    "op_setting_2": "Mn[-]", # Mach number
    "op_setting_3": "TLA[deg]", # Thrust lever angle (detent?)
    "sensor_1": "T2[R]",  # Total temperature at fan inlet 
    "sensor_2": "T24[R]", # Total temperature at LPC outlet
    "sensor_3": "T30[R]", # Total temperature at HPC outlet
    "sensor_4": "T50[R]", # Total temperature at LPT outlet
    "sensor_5": "P2[psi]", # Pressure at fan inlet
    "sensor_6": "P15[psi]", # Total pressure in bypass-duct
    "sensor_7": "P30[psi]", # Total pressure at HPC outlet
    "sensor_8": "Nf[rpm]", # Physical fan speed
    "sensor_9": "Nc[rpm]", # Physical core speed
    "sensor_10": "epr[-]", # Engine pressure ratio (P50/P2)
    "sensor_11": "phi[pph/psi]", # Ratio of fuel flow to Ps30, (not pps/psi, but pph/psi)
    "sensor_12": "Ps30[psi]", # Static pressure at HPC outlet
    "sensor_13": "NRf[rpm]", # Corrected fan speed
    "sensor_14": "NRc[rpm]", # Corrected core speed
    "sensor_15": "BPR[-]", # Bypass Ratio
    "sensor_16": "farB[-]", # Burner fuel-air ratio
    "sensor_17": "htBleed[]",# Bleed Enthalpy
    "sensor_18": "Nf_dmd[rpm]", # Demanded fan speed
    "sensor_19": "PCNfR_dmd[Pct]", # Demanded corrected fan speed
    "sensor_20": "W31[lbm/s]", # HPT coolant bleed
    "sensor_21": "W32[lbm/s]", # LPT coolant bleed
}


train_df = train_df.rename(columns=rename_dict)
test_df  = test_df.rename(columns=rename_dict)

def clean_round(series, ndigits=1, eps=1e-6):
    """Round values and force -0.0 to 0.0"""
    rounded = series.round(ndigits)
    rounded[rounded.abs() < eps] = 0
    return rounded

train_df['condition'] = (
    clean_round(train_df["Alt[kft]"], 0).astype(str) + '_' +
    clean_round(train_df["Mn[-]"], 1).astype(str) + '_' +
    clean_round(train_df["TLA[deg]"], 0).astype(str)
)

train_df

In [ ]:
# [6. 파생 변수 생성 (도메인 지식 기반)]
# - 쓰임새: 엔진의 효율이나 상태를 나타내는 새로운 지표(예: 온도비, 연료 유량)를 계산합니다.
# - 적용이유: 원본 데이터만으로는 파악하기 어려운 엔진의 '건강 상태'를 숫자로 변환하여 모델이 학습할 수 있게 돕습니다.
# - 변경내용: 데이터프레임 우측에 새로운 계산된 컬럼들이 추가됩니다.

# 안전한 계산 함수 (컬럼이 있을 때만 계산)
def safe_div(col1, col2, new_col):
    if col1 in train_df.columns and col2 in train_df.columns:
        train_df[new_col] = train_df[col1] / train_df[col2]
        return True
    return False

# 가능한 조합들만 계산
# LPC 온도비 (T24 / T2) -> T2가 없으면 생략됨
safe_div('T24[R]', 'T2[R]', 'LPC.TR[-]')

# HPC 온도비 (T30 / T24)
safe_div('T30[R]', 'T24[R]', 'HPC.TR[-]')

# 연료 유량 (phi * Ps30)
if 'phi[pph/psi]' in train_df.columns and 'Ps30[psi]' in train_df.columns:
    train_df["Wf[pph]"] = train_df["phi[pph/psi]"] * train_df["Ps30[psi]"]

# 보정된 연료 유량 (T2[R]이 리스트에 없으므로 생략하거나 다른 값 기준)
if 'phi[pph/psi]' in train_df.columns and 'T2[R]' in train_df.columns:
    train_df["WfP3C[pph/psi]"] = train_df["phi[pph/psi]"] / np.sqrt(train_df["T2[R]"] / 518.67)

# 결측치 및 무한대 처리
train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
train_df = train_df.bfill().ffill()

print(f"✅ 사용 가능한 컬럼으로 파생 변수 생성을 완료했습니다!")
print(f"📊 최종 컬럼 수: {len(train_df.columns)}개")

In [ ]:
# [7. 데이터 품질 확인 및 통계 분석]
# - 쓰임새: .head()로 상위 10개 데이터를 보고, .describe()로 평균/최솟값/최댓값 등 통계를 확인합니다.
# - 적용이유: 데이터가 깨지지는 않았는지, 혹은 값이 전혀 변하지 않는 고장 난 센서가 있는지 분석 전 '사전 검사'를 하는 과정입니다.
# - 변경내용: 분석가가 데이터의 전체적인 크기와 숫자 범위를 파악하게 됩니다.

train_df.describe()

In [ ]:
# [8. 결측치 및 이상치 처리]
# - 쓰임새: 데이터에 비어있는 값(NaN)이나 비정상적인 값(무한대 등)을 제거하거나 채웁니다.
# - 적용이유: 모델이 엉뚱한 값을 학습하지 않도록 데이터를 '깨끗하게' 정제하는 필수 과정입니다.
# - 변경내용: 결측치가 제거되거나 대체되어 데이터의 행 수가 줄어들거나 값이 수정됩니다.

train_df.isnull().sum()

In [ ]:
# [9. 데이터프레임의 컬럼 정보 확인]
# - 쓰임새: 데이터프레임의 컬럼 이름, 데이터 타입, 결측치 개수 등을 한눈에 확인합니다.
# - 적용이유: 데이터의 구조를 파악하고 결측치가 있는지 확인하여 전처리 계획을 세우는 데 도움을 줍니다.
# - 변경내용: 데이터프레임의 컬럼 정보가 요약된 표가 생성됩니다.

pd.set_option('display.max_rows', None)
summary_df = pd.DataFrame(train_df.dtypes, columns=['Data Type'])
summary_df = summary_df.reset_index()
summary_df = summary_df.rename(columns={'index': 'Column Name'})
summary_df['Non-Null Count'] = train_df.count().values
summary_df['Null Count'] = train_df.isnull().sum().values
summary_df['Null Ratio (%)'] = (train_df.isnull().sum().values / len(train_df)) * 100
summary_df

### 컬럼 설명 (Data Dictionary)

| 컬럼명 | 설명 | 타입 |
|---|---|---|
| **unit_number** | 엔진 고유 식별자 (Unit Number) | int64 |
| **time_in_cycles** | 운전 사이클 (Time in Cycles) | int64 |
| **Alt[kft]** | 고도 (Altitude) | float64 |
| **Mn[-]** | 마하 수 (Mach Number) | float64 |
| **TLA[deg]** | 스로틀 레버 각도 (Thrust Lever Angle) | float64 |
| **T2[R]** | 팬 입구 전온도 (Total temperature at fan inlet) | float64 |
| **T24[R]** | LPC 출구 전온도 (Total temperature at LPC outlet) | float64 |
| **T30[R]** | HPC 출구 전온도 (Total temperature at HPC outlet) | float64 |
| **T50[R]** | LPT 출구 전온도 (Total temperature at LPT outlet) | float64 |
| **P2[psi]** | 팬 입구 압력 (Pressure at fan inlet) | float64 |
| **P15[psi]** | 바이패스 덕트 전압력 (Total pressure in bypass-duct) | float64 |
| **P30[psi]** | HPC 출구 전압력 (Total pressure at HPC outlet) | float64 |
| **Nf[rpm]** | 물리적 팬 속도 (Physical fan speed) | float64 |
| **Nc[rpm]** | 물리적 코어 속도 (Physical core speed) | float64 |
| **epr[-]** | 엔진 압력비 (Engine pressure ratio) | float64 |
| **phi[pph/psi]** | 연료 유량 대 Ps30 비율 (Ratio of fuel flow to Ps30) | float64 |
| **Ps30[psi]** | HPC 출구 정압 (Static pressure at HPC outlet) | float64 |
| **NRf[rpm]** | 보정된 팬 속도 (Corrected fan speed) | float64 |
| **NRc[rpm]** | 보정된 코어 속도 (Corrected core speed) | float64 |
| **BPR[-]** | 바이패스 비 (Bypass Ratio) | float64 |
| **farB[-]** | 연소기 연료-공기 비 (Burner fuel-air ratio) | float64 |
| **htBleed[]** | 블리드 엔탈피 (Bleed Enthalpy) | int64 |
| **Nf_dmd[rpm]** | 요구 팬 속도 (Demanded fan speed) | int64 |
| **PCNfR_dmd[Pct]** | 요구 보정 팬 속도 (Demanded corrected fan speed) | float64 |
| **W31[lbm/s]** | HPT 냉각 블리드 (HPT coolant bleed) | float64 |
| **W32[lbm/s]** | LPT 냉각 블리드 (LPT coolant bleed) | float64 |
| **condition** | 운전 조건 문자열 (Operational Condition String) | object |

* 참고
| **RUL** | 엔진의 남은 수명 (Remaining Useful Life) | int64 |

In [ ]:
# [10. 컬럼별 데이터 타입 확인]
# - 쓰임새: 데이터프레임의 컬럼 이름, 데이터 타입, 결측치 개수 등을 한눈에 확인합니다.
# - 적용이유: 데이터의 구조를 파악하고 결측치가 있는지 확인하여 전처리 계획을 세우는 데 도움을 줍니다.
# - 변경내용: 데이터프레임의 컬럼 정보가 요약된 표가 생성됩니다.

train_df.info(verbose=True, show_counts=True)

In [ ]:
# [11. 데이터프레임의 컬럼 정보 확인]
# - 쓰임새: 데이터프레임의 컬럼 이름, 데이터 타입, 결측치 개수 등을 한눈에 확인합니다.
# - 적용이유: 데이터의 구조를 파악하고 결측치가 있는지 확인하여 전처리 계획을 세우는 데 도움을 줍니다.
# - 변경내용: 데이터프레임의 컬럼 정보가 요약된 표가 생성됩니다.

train_df.columns

In [ ]:
# [11. 엔진별 수명 시각화]
# - 쓰임새: 각 엔진이 최대 몇 사이클까지 작동했는지 시각적으로 확인합니다.
# - 적용이유: 데이터셋에 포함된 엔진들의 수명 분포를 파악하여, 조기 고장 엔진이 있는지, 수명이 다한 엔진이 있는지 확인합니다.
# - 변경내용: 가로 막대 그래프가 생성되어 엔진 번호와 최대 작동 사이클이 표시됩니다.

max_time_cycles=train_df[['unit_number','time_in_cycles']].groupby('unit_number').max()
plt.figure(figsize=(20,50))
ax=max_time_cycles['time_in_cycles'].plot(kind='barh',width=0.8, stacked=True,align='center')
plt.title('Turbofan Engines LifeTime',fontweight='bold',size=30)
plt.xlabel('Time cycle',fontweight='bold',size=20)
plt.xticks(size=15)
plt.ylabel('Engine ID',fontweight='bold',size=20)
plt.yticks(size=15)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# [12. 최대 작동 사이클 분포 시각화]
# - 쓰임새: 전체 엔진들의 최대 작동 사이클이 어떤 분포를 이루는지 확인합니다.
# - 적용이유: 데이터셋에 포함된 엔진들의 수명 분포를 파악하여, 조기 고장 엔진이 있는지, 수명이 다한 엔진이 있는지 확인합니다.
# - 변경내용: 히스토그램과 커널 밀도 추정(KDE) 곡선이 함께 표시된 그래프가 생성됩니다.

sns.displot(max_time_cycles['time_in_cycles'],kde=True,bins=20,height=6,aspect=2)
plt.xlabel('max time in cycles')

In [ ]:
# [13. 상관관계 히트맵 시각화]
# - 쓰임새: 데이터프레임 내 모든 수치형 컬럼 간의 상관관계를 시각적으로 확인합니다.
# - 적용이유: 어떤 센서들이 서로 관련이 있는지, 또는 특정 센서가 엔진 수명과 얼마나 관련이 있는지 파악하여 피처 엔지니어링이나 모델링에 활용할 수 있습니다.
# - 변경내용: 색상으로 표현된 상관관계 행렬(히트맵)이 생성됩니다.

plt.figure(figsize=(15, 10))
sns.heatmap(train_df.corr(numeric_only=True), annot=True, fmt=".2f", cmap='coolwarm')
plt.show()

In [ ]:
# [14. RUL 컬럼 생성]
# - 쓰임새: 각 엔진별로 '최대 수명 - 현재 시간'을 계산하여 '남은 수명(RUL)' 컬럼을 새로 만듭니다.
# - 적용 이유: 원본 데이터에는 '현재까지 얼마나 운전했는지'만 있고, 우리가 맞춰야 할 '앞으로 얼마나 남았는지'가 없기 때문입니다.
# - 변경 내용: 데이터셋 맨 오른쪽에 인공지능이 맞춰야 할 정답지인 'RUL' 컬럼이 추가됩니다.


# 엔진(unit_number)별로 가장 큰 사이클 번호를 찾습니다.
max_cycle = train_df.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycle.columns = ['unit_number', 'max_cycle']

# 원본 데이터에 최대 수명 정보를 합칩니다.
train_df = train_df.merge(max_cycle, on='unit_number', how='left')

# RUL = 최대 수명 - 현재 사이클
train_df['RUL'] = train_df['max_cycle'] - train_df['time_in_cycles']

# 사용이 끝난 max_cycle 컬럼은 삭제합니다.
train_df.drop('max_cycle', axis=1, inplace=True)

print("RUL 컬럼 생성 완료!")
train_df[['unit_number', 'time_in_cycles', 'RUL']].head()

In [ ]:
# [15. 상관관계 계산 (숫자 데이터만!)]
# - 쓰임새: 데이터프레임 내 모든 수치형 컬럼 간의 상관관계를 시각적으로 확인합니다.
# - 적용이유: 어떤 센서들이 서로 관련이 있는지, 또는 특정 센서가 엔진 수명과 얼마나 관련이 있는지 파악하여 피처 엔지니어링이나 모델링에 활용할 수 있습니다.
# - 변경내용: 색상으로 표현된 상관관계 행렬(히트맵)이 생성됩니다.

plt.figure(figsize=(15, 12))

# RUL과 센서들 간의 상관관계만 뽑아서 보기 좋게 정렬합니다.
correlations = train_df.corr(numeric_only=True)['RUL'].sort_values(ascending=False)

# 히트맵 그리기
sns.heatmap(train_df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap (User Index applied)', fontsize=20)
plt.show()

print("--- RUL과 상관관계가 높은 순서 ---")
print(correlations)

In [ ]:
# [16. 데이터 정제 - 노이즈 변수 및 오차 제거]
# - 쓰임새: 값이 변하지 않는 상수 컬럼(표준편차 0)을 삭제하고, 컴퓨터 연산 오차(부동소수점)를 다듬습니다.
# - 적용 이유: 변하지 않는 값은 학습에 도움이 안 되고 계산 속도만 늦추기 때문입니다.
# - 변경 내용: 학습 효율이 떨어지는 불필요한 데이터가 삭제되어 데이터가 가벼워지고 깨끗해집니다.

# 🔍 값이 변하지 않는(분산이 0인) 무의미한 컬럼 자동 추출
constant_columns = [col for col in train_df.columns if train_df[col].nunique() <= 1]

# 📝 자동으로 찾은 고정값 컬럼에 'sensor_6'와 'op_setting_3'를 추가
cols_to_drop = list(set(constant_columns + ['sensor_6', 'op_setting_3']))

# ✂️ 데이터프레임에서 불필요한 컬럼 삭제 실행
train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f"❌ 삭제된 컬럼 목록: {cols_to_drop}")
print(f"✅ 남은 컬럼 개수: {len(train_df.columns)}개")
print(f"📊 현재 컬럼 목록: {train_df.columns.tolist()}")

In [ ]:
# [17. 데이터 정제 - 노이즈 변수 및 오차 제거]
# - 쓰임새: 값이 변하지 않는 상수 컬럼(표준편차 0)을 삭제하고, 컴퓨터 연산 오차(부동소수점)를 다듬습니다.
# - 적용 이유: 변하지 않는 값은 학습에 도움이 안 되고 계산 속도만 늦추기 때문입니다.
# - 변경 내용: 학습 효율이 떨어지는 불필요한 데이터가 삭제되어 데이터가 가벼워지고 깨끗해집니다.

plt.figure(figsize=(12, 10))
sns.heatmap(train_df.corr(numeric_only=True), annot=True, cmap='RdYlGn', fmt=".2f")
plt.title("Cleaned Data Correlation")
plt.show()

In [ ]:
# [18. 이동평균을 이용한 피처 엔지니어링]
# - 쓰임새: 최근 5개 시점의 데이터 평균을 구하는 '이동 평균(Moving Average)' 기법을 적용합니다.
# - 적용 이유: 센서 데이터는 미세한 진동(노이즈)이 심해, 그대로 사용하면 AI가 패턴을 찾기 어렵습니다. 노이즈를 지우고 '경향성'을 살리기 위함입니다.
# - 변경 내용: 심하게 튀던 그래프가 부드러운 곡선으로 바뀌며, 고장 시점에 따른 센서의 상승/하락 추세가 선명해집니다.


# 🔍 분석 대상 센서 리스트 (상관관계가 높았던 주요 센서들)
main_sensors = [
    'T24[R]', 'T30[R]', 'T50[R]', 'P30[psi]', 'Nf[rpm]', 
    'Nc[rpm]', 'Ps30[psi]', 'NRf[rpm]', 'NRc[rpm]', 'BPR[-]', 
    'W31[lbm/s]', 'W32[lbm/s]'
]

window_size = 5

# 🌊 5개 사이클씩 묶어서 평균 계산 (이동 평균)
for s in main_sensors:
    if s in train_df.columns:
        # A. 이동 평균 (Moving Average)
        train_df[f'{s}_avg'] = train_df.groupby('unit_number')[s].transform(lambda x: x.rolling(window=window_size).mean())
        # B. 표준편차 (Rolling Std)
        train_df[f'{s}_std'] = train_df.groupby('unit_number')[s].transform(lambda x: x.rolling(window=window_size).std())
        # C. 변화량 (Difference)
        train_df[f'{s}_diff'] = train_df.groupby('unit_number')[s].transform(lambda x: x.diff(periods=window_size))

# 결측치 처리 (NaN 제거)
train_df = train_df.bfill()

print("✅ 평균(_avg), 표준편차(_std), 기울기(_diff) 생성 및 결측치 처리 완료!")

In [ ]:
# [19. 누적 성능 변화량 계산]
# - 쓰임새: "지금까지 비행하면서 성능이 얼마나 나빠졌는가?"를 수치화합니다.
# - 적용 이유: 엔진은 시간이 지날수록 마모되어 성능이 떨어집니다. 이 '누적된 손상도'가 고장을 예측하는 가장 강력한 신호입니다.
# - 변경 내용: 각 센서별로 '최고 성능(최소값)'을 기록해두고, 현재 성능과의 차이를 계산하여 새로운 피처로 추가합니다.


# 📉 "가장 좋았을 때보다 얼마나 변했나?" 계산
for sensor in main_sensors:
    if sensor in train_df.columns:
        # 현재까지의 최대값 찾기
        cum_max = train_df.groupby('unit_number')[sensor].cummax()
        # 최대값과 현재값의 차이 계산
        train_df[f'{sensor}_diff'] = cum_max - train_df[sensor]

# 🛠️ 앞부분에 생기는 빈칸(NaN) 채우기
train_df = train_df.infer_objects(copy=False).fillna(method='bfill')

print("✅ 성능 변화량(_diff) 변수 생성 완료!")
print(train_df.head())

In [ ]:
# [20. 피처 엔지니어링 결과 시각화 및 상관관계 분석]
# - 쓰임새: 새로 만든 '평균(avg)'과 '변화량(diff)' 피처가 실제로 RUL과 얼마나 관련이 있는지 시각적으로 확인합니다.
# - 적용 이유: 상관관계가 높을수록 해당 피처가 고장 예측에 유용하다는 뜻이므로, 모델 학습 시 우선순위를 높일 수 있습니다.
# - 변경 내용: 히트맵을 통해 어떤 센서의 어떤 변화가 고장과 밀접한지 한눈에 파악할 수 있습니다.

new_cols = [col for col in train_df.columns if '_avg' in col or '_diff' in col] + ['RUL']
plt.figure(figsize=(10, 8))
sns.heatmap(train_df[new_cols].corr(numeric_only=True)[['RUL']].sort_values(by='RUL', ascending=False), 
            annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation: Engineered Features vs RUL")
plt.show()

In [ ]:
# [21. 요약 통계량 보기]
# - 쓰임새: 새로 만든 피처들의 분포와 중심 경향성을 확인합니다.
# - 적용 이유: 평균, 표준편차, 최소/최대값 등을 통해 데이터의 특성을 파악하고, 스케일링이나 이상치 탐지의 기초 자료로 활용합니다.
# - 변경 내용: describe() 메서드를 사용하여 요약 통계량을 출력합니다.

train_df[new_cols].describe()

In [ ]:
# [22. RUL 클리핑 (Piecewise Linear)]
# - 쓰임새: RUL의 최대값을 특정 값(예: 130)으로 제한하여 모델이 초반에 헷갈리지 않게 합니다.
# - 적용 이유: 실제 비행 데이터에서는 고장 직전까지 RUL이 거의 선형적으로 감소하다가 마지막에 급격히 떨어지는 경향이 있습니다. 이를 모델에 반영하기 위해 일정 주기 이상은 최대값으로 고정합니다.
# - 변경 내용: apply 함수를 사용하여 RUL 값이 130보다 크면 130으로 변경

train_df['RUL'] = train_df['RUL'].apply(lambda x: 130 if x > 130 else x)

sns.lineplot(data=train_df[train_df['unit_number']==1], x='time_in_cycles', y='RUL')
plt.title("Clipped RUL (Piecewise Linear)")
plt.show()

In [ ]:
# [23. 이동 평균 및 변화량, 상호작용 피처 생성]
# - 쓰임새: 센서 데이터의 노이즈를 줄이고(이동 평균), 변화 추세(변화량)와 센서 간의 관계(상호작용)를 파악합니다.
# - 적용 이유: 고장 예측 시점의 단순한 수치뿐만 아니라, 최근의 변화 추세와 다른 센서와의 복합적인 상태를 고려해야 더 정확한 예측이 가능합니다.
# - 변경 내용: rolling().mean(), rolling().std(), diff(), 그리고 T24와 P30의 비율 계산

exclude_cols = ['unit_number', 'time_in_cycles', 'condition', 'RUL']

# 현재 데이터프레임에서 실제 센서 컬럼들만 리스트로 만듭니다.
# (숫자형 데이터이면서 제외 목록에 없는 컬럼만 추출)
# 계산 대상 센서 리스트 (기존 변수 활용)
actual_sensors = [c for c in train_df.columns if c in col_names and c not in index_names + setting_names]

window_size = 5

for s in actual_sensors:
    # (기존) 이동 평균 및 표준편차
    train_df[f'{s}_avg'] = train_df.groupby('unit_number')[s].transform(lambda x: x.rolling(window=window_size).mean())
    train_df[f'{s}_std'] = train_df.groupby('unit_number')[s].transform(lambda x: x.rolling(window=window_size).std())
    
    # (추가 1) 기울기(Gradient): 5사이클 전과 현재의 차이
    # "얼마나 빨리 변하고 있는가"를 측정합니다.
    train_df[f'{s}_diff'] = train_df.groupby('unit_number')[s].transform(lambda x: x.diff(periods=window_size))

# (추가 2) 상호작용(Interaction): 온도(T24)와 압력(P30)의 비율
# 엔진의 효율 저하를 더 민감하게 포착합니다. (컬럼명이 존재할 경우에만 생성)
if 'T24[R]' in train_df.columns and 'P30[psi]' in train_df.columns:
    train_df['T24_P30_ratio'] = train_df['T24[R]'] / (train_df['P30[psi]'] + 1e-9)

# 결측치 채우기
train_df = train_df.bfill()

print(f"✅ 모든 센서에 대한 이동 평균(_avg) 변수 생성 완료!")
print(f"현재 총 컬럼 수: {len(train_df.columns)}개")

In [ ]:
# [24. 학습 데이터 준비 (Feature Selection)]
# - 쓰임새: 모델이 학습할 피처(Feature)와 타겟(Target)을 분리하고, 불필요한 컬럼을 제외합니다.
# - 적용 이유: 'unit_number'나 'time_in_cycles' 같은 ID 정보는 예측과 관련이 없으므로 학습에서 제외해야 합니다.
# - 변경 내용: exclude 리스트를 정의하여 학습에 사용할 컬럼만 선택


# 1. 원본 센서들 + 이동평균 센서들만 리스트로 만들기
# 'unit_number', 'time_in_cycles', 'RUL' 처럼 공부하면 안 되는 것들을 제외합니다.
exclude = ['unit_number', 'time_in_cycles', 'RUL', 'condition']
target_sensors = [c for c in train_df.columns if c not in exclude]

print(f"✅ 모델이 공부할 총 변수 개수: {len(target_sensors)}개")

In [ ]:
# [25. 이동평균 시각화]
# - 쓰임새: 이동평균이 원본 데이터의 노이즈를 얼마나 잘 줄였는지 시각적으로 확인합니다.
# - 적용 이유: 그래프를 통해 이동평균이 실제로 더 부드러운 추세를 보여주는지 확인하여, 피처 생성의 효과를 검증합니다.
# - 변경 내용: unit_number가 1인 데이터만 추출하여 원본 데이터와 이동평균 데이터를 함께 그래프로 그림
#윈도우 사이즈 5로 이동평균 계산

# 1. 데이터 추출
unit_1 = train_df[train_df['unit_number'] == 1].copy()

# --- 첫 번째 그래프: P30[psi] 상세 분석 --- # 원본 데이터 (회색)
plt.figure(figsize=(15, 5))
plt.plot(unit_1['time_in_cycles'], unit_1['P30[psi]'], label='Original P30[psi]', color='lightgray', alpha=0.8)
plt.plot(unit_1['time_in_cycles'], unit_1['P30[psi]_avg'].fillna(method='bfill'), label='Moving Average', color='red', linewidth=2)
plt.title('Graph 1: Sensor P30[psi] Analysis (Original vs Moving Average)', fontsize=13)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# --- 두 번째 그래프: T24[R] 상세 분석 (아까 sensor_11과 같은 것) --- # 원본 데이터 (회색)
plt.figure(figsize=(15, 5))
plt.plot(unit_1['time_in_cycles'], unit_1['T24[R]'], label='Original T24[R]', color='lightgray', alpha=0.8)
plt.plot(unit_1['time_in_cycles'], unit_1['T24[R]_avg'].fillna(method='bfill'), label='Moving Average', color='blue', linewidth=2)
plt.title('Graph 2: Sensor T24[R] Analysis (Original vs Moving Average)', fontsize=13)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# [26. 이동평균과 표준편차 생성]
# - 쓰임새: 이동평균은 데이터의 노이즈를 줄여 추세를 명확하게 하고, 표준편차는 데이터의 변동성(불안정성)을 수치화합니다.
# - 적용 이유: 두 피처를 함께 사용하면 모델이 엔진의 '평균적인 상태'와 '현재 얼마나 불안정한지'를 동시에 파악할 수 있어 예측 성능이 향상됩니다.
# - 변경 내용: 기존 센서 컬럼들에 대해 이동평균(_avg)과 표준편차(_std)를 계산하여 데이터프레임에 추가

# 1. 계산에서 제외할 컬럼 (기본 정보 및 정답)
exclude_cols = ['unit_number', 'time_in_cycles', 'RUL', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'condition']

# 2. 실제 센서 컬럼들만 리스트로 추출
# (현재 train_df에 존재하는 컬럼 중 제외 목록에 없는 것들)
actual_sensors = [c for c in train_df.columns if c not in exclude_cols and '_avg' not in c and '_std' not in c]

window_size = 5

for s in actual_sensors:
    # A. 이동 평균 (Rolling Mean) 생성
    train_df[f'{s}_avg'] = train_df.groupby('unit_number')[s].transform(lambda x: x.rolling(window=window_size).mean())
    
    # B. 표준편차 (Rolling Std) 생성 👈 이게 추가된 부분!
    train_df[f'{s}_std'] = train_df.groupby('unit_number')[s].transform(lambda x: x.rolling(window=window_size).std())

# 3. 결측치 채우기 (초반 5개 행의 NaN 제거)
train_df = train_df.bfill()

print(f"✅ {len(actual_sensors)}개 센서에 대해 평균(_avg)과 표준편차(_std) 생성 완료!")

In [ ]:
# [27. 데이터 스케일링 - 단위 통일]
# - 쓰임새: 모든 센서 데이터의 범위를 0과 1 사이로 압축합니다.
# - 적용 이유: 온도(600도)와 압력(15psi)처럼 단위 차이가 크면, 모델이 숫자가 큰 온도만 중요하다고 착각할 수 있습니다. 공정한 학습을 위해 단위를 맞춥니다.
# - 변경 내용: 서로 제각각이던 데이터의 크기가 0~1 사이로 균일하게 맞춰집니다.

exclude_cols = ['unit_number', 'time_in_cycles', 'RUL', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'condition']
scaling_cols = [c for c in train_df.columns if c not in exclude_cols and train_df[c].dtype != 'object']

scaler = MinMaxScaler()
train_df[scaling_cols] = scaler.fit_transform(train_df[scaling_cols])

print(f"✨ 스케일링 완료! 대상 컬럼 수: {len(scaling_cols)}개")
print(f"📋 첫 5개 스케일링 컬럼 예시: {scaling_cols[:5]}")

In [ ]:
# [28. 엔진 상태 세분화]
# - 쓰임새: 엔진의 상태를 '정상', '약간 불안정', '심하게 불안정' 등 여러 단계로 나눕니다.
# - 적용 이유: 단순한 수치(예: 100도)만 보는 것이 아니라, '평소보다 얼마나 변했는가'를 기준으로 엔진의 건강 상태를 더 정확하게 진단할 수 있습니다.
# - 변경 내용: is_vibrating(떨림 여부)과 stress_index(종합 위험 지수)라는 새로운 컬럼이 추가됩니다.


std_mean = train_df['T24[R]_std'].mean()

# 평균보다 떨림이 크면 1(위험 징후), 낮으면 0(안정)으로 세분화
train_df['is_vibrating'] = (train_df['T24[R]_std'] > std_mean).astype(int)

#(선택사항) 여러 센서의 떨림을 합쳐서 '종합 위험 지수' 만들기
# 떨림이 발생하는 센서가 많을수록 숫자가 높아집니다.
std_cols = [c for c in train_df.columns if '_std' in c]
train_df['stress_index'] = train_df[std_cols].gt(train_df[std_cols].mean()).sum(axis=1)

print("✅ 엔진 상태 세분화 인덱스(is_vibrating, stress_index) 생성 완료!")

In [ ]:
# [29. 학습 데이터셋 구성]
# - 쓰임새: 모델이 공부할 '문제지(X)'와 '정답지(y)'를 만듭니다.
# - 적용 이유: 엔진 번호나 정답(RUL) 같은 불필요한 정보가 섞여 있으면 모델이 헷갈리기 때문에, 학습에 꼭 필요한 센서 데이터만 골라냅니다.
# - 변경 내용: features 리스트에 학습에 사용할 컬럼명들이 저장됩니다.

# 정답(RUL)이나 엔진번호처럼 학습에 쓰면 안 되는 것들을 제외합니다.
exclude_cols = ['unit_number', 'time_in_cycles', 'RUL', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'condition']
features = [c for c in train_df.columns if c not in exclude_cols]

print(f"✅ 학습에 사용될 피처 개수: {len(features)}개")
print(f"📊 포함된 피처들: {features}")

In [ ]:
# [30. 데이터 스케일링]
# - 쓰임새: 모든 센서 데이터의 눈높이를 맞춰줍니다.
# - 변경 내용: scaling_cols 리스트에 스케일링할 컬럼명들이 저장됩니다.

train_df[scaling_cols].describe()

In [ ]:
# [31. 데이터 시각화]
# - 쓰임새: 각 센서가 고장나기 전까지 어떻게 변하는지 눈으로 확인합니다.
# - 적용 이유: 그래프를 보면 '아, 이 센서는 100이 넘어가면 위험하구나' 하는 감을 잡을 수 있습니다.
# - 변경 내용: 히스토그램(막대그래프)을 그려서 데이터 분포를 확인합니다.


# 1. 📋 분석에서 제외할 기본 정보 컬럼들
exclude_cols = ['unit_number', 'time_in_cycles', 'RUL']

# 2. 🔍 나머지 실제 센서 데이터 컬럼들만 쏙 골라내기
actual_sensors = [c for c in train_df.columns if c not in exclude_cols]

print(f"✅ 분석할 센서들을 찾았습니다: {actual_sensors[:5]}... 등")

# 3. 📊 히스토그램 그리기 (상위 9개 센서)
if len(actual_sensors) == 0:
    print("❌ 여전히 컬럼을 찾을 수 없습니다. 데이터 로드 부분을 확인해주세요.")
else:
    # 한 화면에 9개까지 그려서 확인해봅시다
    train_df[actual_sensors[:9]].hist(bins=20, figsize=(15, 12), color='lightcoral', edgecolor='black')
    plt.suptitle("Sensor Physical Value Distribution", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
# [32. 데이터 정제]
# - 쓰임새: 결측치나 이상치를 제거하여 데이터 품질을 높입니다.
# - 적용 이유: 결측치가 있으면 모델이 학습을 못하고, 이상치가 있으면 오작동할 수 있습니다.
# - 변경 내용: 결측치를 0으로 채우고, 이상치를 제거합니다.


# 1. 빈 값이 있는지 확인 (0이어야 함)
print("❌ 결측치 개수:\n", train_df.isnull().sum().sum())

# 2. 모든 데이터가 숫자인지 확인 (object가 없어야 함)
print("\n📋 데이터 타입 요약:")
print(train_df.dtypes.value_counts())

In [ ]:
# [33. 파생변수 생성]
# - 쓰임새: 기존 데이터만으로는 부족한 정보를 새롭게 만들어냅니다.
# - 적용 이유: 센서 값 자체보다 '변화량'이나 '떨림'이 고장을 더 잘 예측할 수 있기 때문입니다.
# - 변경 내용: 'is_vibrating' 컬럼이 추가되었습니다.

std_threshold = train_df['T24[R]_std'].mean() * 2
train_df['is_vibrating'] = (train_df['T24[R]_std'] > std_threshold).astype(int)

In [ ]:
# [34. 데이터 분할]
# - 쓰임새: 모델을 학습시킬 데이터와 평가할 데이터를 분리합니다.
# - 적용 이유: 같은 엔진으로 학습하고 평가하면 '기계가 외운 것'인지 '진짜 실력'인지 알 수 없기 때문에, 엔진 번호를 기준으로 섞어서 분리합니다.
# - 변경 내용: train_set과 val_set이 생성되었습니다.

gss = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=42)
train_idx, val_idx = next(gss.split(train_df, groups=train_df['unit_number']))

train_set = train_df.iloc[train_idx]
val_set = train_df.iloc[val_idx]
print(f"✅ 학습용 분할 완료!")
print(f"학습용 엔진: {train_set['unit_number'].nunique()}개, 검증용 엔진: {val_set['unit_number'].nunique()}개")

In [ ]:
# [35. 피처 선택]
# - 쓰임새: 모델이 학습할 데이터(X)와 정답(y)을 분리합니다.
# - 적용 이유: 'unit_number'나 'RUL' 같은 정답/분류 정보는 모델이 보면 안 되므로 제외합니다.
# - 변경 내용: X_train, y_train, X_val, y_val이 생성되었습니다.


exclude_cols = ['unit_number', 'time_in_cycles', 'RUL', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'condition']

# 'condition'이 혹시 없을 수도 있으니 안전하게 필터링
features = [c for c in train_df.columns if c not in exclude_cols]

X_train = train_set[features]
y_train = train_set['RUL']
X_val = val_set[features]
y_val = val_set['RUL']

print(f"✅ 학습 준비 완료! 사용된 피처 수: {len(features)}개")

### 모델 팀 결성, 학습, 예측, 평가

In [ ]:
# [36. 앙상블 모델]
# --- [1단계] 컬럼명 수정 ---
# - 쓰임새: XGBoost 모델이 정상적으로 작동하기 위해 컬럼명을 수정합니다.
# - 적용이유: XGBoost는 특정 문자(예: '[', ']', '<')를 컬럼명으로 인정하지 않습니다.
# - 변경내용: 컬럼명을 '_lt'로 변경하여 XGBoost 호환 모드로 전환합니다.

def clean_column_names(df):
    new_cols = [col.replace('[', '_').replace(']', '').replace('<', 'lt') for col in df.columns]
    df.columns = new_cols
    return df

X_train = clean_column_names(X_train)
X_val = clean_column_names(X_val)

print("✅ 컬럼명 수정 완료! (XGBoost 호환 모드)")

In [ ]:
# [37. 앙상블 모델]
# - 쓰임새: 개별 모델을 정의하여 앙상블 모델을 학습합니다.
# - 적용이유: 개별 모델의 성능을 개별적으로 평가하고, 앙상블 모델의 성능을 향상시킵니다.
# - 변경내용: 개별 모델을 정의하여 앙상블 모델을 학습합니다.


# 가장 단순한 모델로, 센서 수치와 수명 사이의 직선적인 관계를 찾습니다. 특별한 기교 없이 원본 그대로의 추세를 보기 위해 사용했습니다.
model_lr = LinearRegression()

model_ridge = Ridge(alpha=0.5)
model_lasso = Lasso(alpha=0.01)

# 2. Decision Tree (의사결정나무)
# max_depth: 나무의 깊이. 너무 깊으면 외워버리고(과적합), 너무 낮으면 공부를 안 합니다.
model_dt = DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42)

# 3. Random Forest (랜덤 포레스트)
# n_estimators: 나무의 개수. 많을수록 안정적이지만 느려집니다.
# max_features: 한 나무가 공부할 때 참고할 센서의 개수 제한.
model_rf = RandomForestRegressor(
    n_estimators=200, 
    max_depth=12, 
    min_samples_split=5, 
    n_jobs=-1, # 내 컴퓨터의 모든 CPU를 다 써서 속도를 높임
    random_state=42
)

# 4. XGBoost (성능 깡패)
# learning_rate: 학습 속도. 보통 0.01~0.1 사이를 씁니다.
# subsample: 데이터 중 일부만 무작위로 골라 학습 (일반화 성능 향상)
model_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# 5. SVR (서포트 벡터 회귀)
# C: 오차를 얼마나 허용할지 결정. (크면 엄격함, 작으면 관대함)
# epsilon: 이 범위 안의 오차는 무시함.
model_svr = SVR(kernel='rbf', C=10.0, epsilon=0.1)

# 6. MLP (신경망)
# hidden_layer_sizes: 뉴런의 개수. (100, 50)는 1층에 100개, 2층에 50개를 의미합니다.
# activation: 활성화 함수. 'relu'는 비선형성을 추가하여 복잡한 패턴을 학습하게 합니다.
# max_iter: 최대 반복 횟수.
model_mlp = MLPRegressor(
    hidden_layer_sizes=(100, 50), # 두 개의 층(100개 노드, 50개 노드)으로 구성된 신경망
    activation='relu',            # 딥러닝의 표준 활성화 함수
    solver='adam',                # 최적화 알고리즘
    max_iter=500,                 # 최대 학습 횟수
    random_state=42
)

In [ ]:
# [38. 앙상블 팀 결성]
# --- [3단계] 앙상블 팀 결성 (전체 투입!) ---
# - 쓰임새: 개별 모델을 정의하여 앙상블 모델을 학습합니다.
# - 적용이유: 개별 모델의 성능을 개별적으로 평가하고, 앙상블 모델의 성능을 향상시킵니다.
# - 변경내용: 개별 모델을 정의하여 앙상블 모델을 학습합니다.

ensemble_all = VotingRegressor(
    estimators=[
        ('lr', model_lr), ('ridge', model_ridge), ('lasso', model_lasso),
        ('dt', model_dt), ('rf', model_rf), ('xgb', model_xgb), 
        ('svr', model_svr), ('mlp', model_mlp) # 딥러닝 멤버 합류!
    ],
    weights=[1, 1, 1, 1, 3, 5, 1, 2] # 딥러닝에게도 적당한 비중(2)을 줍니다.
)

In [ ]:
# [39. 앙상블 모델 학습]
# - 쓰임새: 앙상블 모델을 학습합니다.
# - 적용이유: 개별 모델의 성능을 개별적으로 평가하고, 앙상블 모델의 성능을 향상시킵니다.
# - 변경내용: 앙상블 모델을 학습합니다.

print("🏋️ 모든 정통 머신러닝 모델 학습 중... 잠시만 기다려 주세요.")
ensemble_all.fit(X_train, y_train)

In [ ]:
# [40. 앙상블 모델 평가]
# - 쓰임새: 앙상블 모델의 성능을 평가합니다.
# - 적용이유: 개별 모델의 성능을 개별적으로 평가하고, 앙상블 모델의 성능을 향상시킵니다.
# - 변경내용: 앙상블 모델의 성능을 평가합니다.

y_pred = ensemble_all.predict(X_val)
rmse_total = np.sqrt(mean_squared_error(y_val, y_pred))

print("\n" + "="*50)
print(f"🏆 전체 앙상블 모델 최종 RMSE: {rmse_total:.4f}")
print("="*50)

In [ ]:
# --- [5단계] 개별 모델별 성적표 출력 ---
print("\n📊 모델별 개별 성적표 (낮을수록 좋음):")
for name, model in ensemble_all.named_estimators_.items():
    pred = model.predict(X_val)
    individual_rmse = np.sqrt(mean_squared_error(y_val, pred))
    print(f"📍 {name:8} 모델 RMSE: {individual_rmse:.4f}")

In [ ]:
# [41. 끝판왕 등장: AutoGluon 자동화 딥러닝/앙상블]
# --- [0단계] AutoGluon 설치 (최초 1회만 실행) ---
# !pip install autogluon

# --- [1단계: 오토글론 전용 데이터셋 준비] ---
# 오토글론은 문제지와 답지가 합쳐진 '표' 형태를 입력으로 받습니다.
train_data = X_train.copy()
train_data['target'] = y_train

val_data = X_val.copy()
val_data['target'] = y_val

# --- [2단계: 인공지능 지휘관(AutoGluon) 학습 시작] ---
# - presets='best_quality': 딥러닝, 스태킹 등 모든 기술을 동원해 최상의 모델을 찾습니다.
# - time_limit=600: 최적의 조합을 찾기 위해 딱 10분(600초)만 투자합니다.
predictor = TabularPredictor(
    label='target', 
    problem_type='regression', 
    eval_metric='rmse'
).fit(
    train_data, 
    presets='best_quality', 
    time_limit=600
)

# --- [3단계: 최종 결과 평가] ---
performance = predictor.evaluate(val_data)
print(f"🚀 오토글론 최종 RMSE: {performance['rmse']}")

# --- [4단계: 모델들이 어떻게 쌓였는지 확인(리더보드)] ---
leaderboard = predictor.leaderboard(val_data)
display(leaderboard)

In [ ]:
# [42. 최종 실전 검증: 외부 Test 문서 & NASA 정답지 비교]
# - 쓰임새: 학습에 전혀 사용되지 않은 외부 문서(Test)와 진짜 정답(RUL)을 대조합니다.
# - 적용이유: 모델의 일반화 성능을 최종적으로 확정하고, 실전 투입 가능성을 검증하기 위함입니다.

# 1. 외부 Test 문서의 엔진별 마지막 상태(현재 시점) 추출
X_test_last = X_test.groupby('unit_nr').last().reset_index()

# 2. 오토글론으로 '실전 예측' 수행
y_test_pred = predictor.predict(X_test_last)

# 3. NASA 공식 정답지(RUL_FD001.txt)와 일대일 비교 채점
from sklearn.metrics import mean_squared_error
import numpy as np

actual_rul = y_test_truth.values.flatten()
predicted_rul = y_test_pred.values.flatten()

# 실전 RMSE 계산
final_real_rmse = np.sqrt(mean_squared_error(actual_rul, predicted_rul))

print("\n" + "="*50)
print(f"🏆 [최종 실전 성적] NASA RUL 대조 RMSE: {final_real_rmse:.4f}")
print("="*50)

# 4. 실전 결과 시각화 (정답 vs 예측)
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))
plt.plot(actual_rul, label='Actual Truth (NASA RUL)', color='#003366', marker='o', markersize=4, alpha=0.8)
plt.plot(predicted_rul, label='AI Prediction', color='#FF3300', linestyle='--', marker='x', markersize=5, alpha=0.8)

plt.title(f'Final Project Result: Actual vs AI (RMSE: {final_real_rmse:.4f})', fontsize=16)
plt.xlabel('Engine Unit Number', fontsize=12)
plt.ylabel('Remaining Useful Life (Cycles)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()